# 🧠 Texturizar TU modelo 3D con TU imagen — MV-Adapter (versión pulida)

Le das **tu malla 3D existente** (el `.glb` de Hunyuan) + **tu imagen**, y la IA genera la textura
**alrededor de todo el modelo** (6 vistas consistentes) y la hornea sobre tu malla → `.glb` texturizado.
**Tu geometría no se toca**: solo le agrega el color.

Esta versión trae la auditoría completa de todos los errores que ya sufrimos con Hunyuan/SF3D, resueltos de antemano:
- ✅ **Sin token ni cuentas** (todos los modelos que baja son públicos).
- ✅ **Sin reiniciar sesión** (la instalación no toca nada que esté cargado en memoria).
- ✅ Baja solos los **pesos extra** que el repo pide aparte (RealESRGAN + LaMa) — sin esto moriría al generar.
- ✅ Usa la variante **SD2.1 (<10 GB de VRAM)** → cómoda en la T4 (la variante SDXL pide 14 GB y va al límite).
- ✅ numpy/scipy pineados a la combinación que ya validamos en Colab.
- ✅ Subida de archivos con plan B para celular (panel Archivos 📁).

## Orden — 5 celdas, en orden, sin saltear
1. GPU: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.
2. **Celda 1** — instalar todo (~8-12 min: compila nvdiffrast y baja pesos). **Una sola vez.**
3. **Celda 2** — subir tu **modelo 3D** (`.glb`).
4. **Celda 3** — subir tu **imagen** (de frente, mejor PNG sin fondo).
5. **Celda 4** — texturizar (la 1ª vez baja el modelo de difusión, varios GB).
6. **Celda 5** — descargar el `.glb` texturizado.

⚠️ Si Colab se desconecta o reinicia solo, `/content` se borra → volvé a la **Celda 1** (más rápida la 2ª vez, queda cache de pip).

## Celda 1 — Instalar TODO (sin reiniciar sesión después)
Tarda ~8-12 min. Al final imprime un chequeo: si dice `TODO OK`, seguí. Los warnings amarillos de pip (numba/cudf/etc.) son inofensivos.

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/MV-Adapter'):
    !git clone https://github.com/huanngzh/MV-Adapter.git
os.chdir('/content/MV-Adapter')

# ninja acelera/destraba la compilacion de nvdiffrast
!apt-get -qq install -y ninja-build > /dev/null 2>&1

# dependencias del repo (incluye nvdiffrast desde git: compila unos minutos)
!pip install -q -r requirements.txt 2>&1 | tail -4

# rembg por si --remove_bg lo usa (leccion de SF3D: no viene en requirements)
!pip install -q rembg onnxruntime 2>&1 | tail -2

# PIN FINAL de numpy/scipy: la combinacion que valida con todo
# (rembg pide numpy>=2.3; numba/cupy de Colab piden numpy<2.5-2.6)
!pip install -q --force-reinstall "numpy>=2.3.0,<2.5" "scipy>=1.16.3" 2>&1 | tail -2

# pesos extra que el repo pide bajar aparte (sin esto, la Celda 4 falla)
os.makedirs('checkpoints', exist_ok=True)
if not os.path.exists('checkpoints/RealESRGAN_x2plus.pth'):
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth -O checkpoints/RealESRGAN_x2plus.pth
if not os.path.exists('checkpoints/big-lama.pt'):
    !wget -q https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt -O checkpoints/big-lama.pt
print('pesos:', round(os.path.getsize('checkpoints/RealESRGAN_x2plus.pth')/1e6,1), 'MB (ESRGAN) |',
      round(os.path.getsize('checkpoints/big-lama.pt')/1e6,1), 'MB (LaMa)')

# Chequeo final en un proceso aparte (no ensucia la memoria del notebook -> no hay que reiniciar)
!python -c "import torch, numpy, scipy, diffusers, transformers, nvdiffrast, open3d, pymeshlab, trimesh; print('torch', torch.__version__, '| GPU', torch.cuda.is_available()); print('numpy', numpy.__version__, '| scipy', scipy.__version__); print('TODO OK ✅  ->  segui con la Celda 2')"

## Celda 2 — Subir tu MODELO 3D (`.glb` / `.obj`)
Tu malla de Hunyuan (la gris). Si el botón no anda en el celular: subilo por el panel **Archivos** 📁 (carpeta a la izquierda) y corré esta celda igual — lo detecta solo.

In [ ]:
import os, glob
assert os.path.isdir('/content/MV-Adapter'), '⚠️ La sesión se reinició y se borró todo: volvé a correr la Celda 1.'

MESH = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        MESH = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not MESH or not os.path.exists(MESH):
    cand = glob.glob('/content/*.glb') + glob.glob('/content/*.obj')
    cand = [c for c in cand if '/MV-Adapter/' not in c]
    cand.sort(key=os.path.getmtime)
    MESH = cand[-1] if cand else None

assert MESH and os.path.exists(MESH), 'No encontré el modelo. Subilo por el botón o por el panel Archivos 📁 y volvé a correr esta celda.'

import trimesh
try:
    _m = trimesh.load(MESH, force='mesh')
    print('Modelo:', MESH, '|', len(_m.faces), 'caras')
    if len(_m.faces) > 150000:
        print('ℹ️ Malla grande: el texturizado puede tardar bastante más (es normal, dejalo correr).')
except Exception as e:
    print('Modelo:', MESH, '(no pude leer el conteo de caras:', e, ')')
print('✅ Seguí con la Celda 3.')

## Celda 3 — Subir tu IMAGEN de referencia
La del personaje **de frente** (mejor PNG sin fondo; `--remove_bg` igual lo quita solo).

In [ ]:
import os, glob
from PIL import Image

IMG = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré la imagen. Subila y volvé a correr esta celda.'
im = Image.open(IMG)
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Seguí con la Celda 4.')

## Celda 4 — Texturizar tu malla con la IA 🧠
Usa la variante **SD2.1** (<10 GB → cómoda en la T4). La **primera vez baja varios GB** de modelos — paciencia.
El resultado queda en `outputs/` como `.glb` texturizado.
> Si algún día corrés esto en una GPU grande (L4/A100), borrá `--variant sd21` del comando para usar SDXL (más calidad, 14 GB).

In [ ]:
import os
assert os.path.isdir('/content/MV-Adapter'), '⚠️ La sesión se reinició: volvé a correr la Celda 1 (y las 2-3).'
assert 'MESH' in dir() and os.path.exists(MESH), '⚠️ Falta el modelo: corré la Celda 2.'
assert 'IMG' in dir() and os.path.exists(IMG), '⚠️ Falta la imagen: corré la Celda 3.'
os.chdir('/content/MV-Adapter')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # menos fragmentacion de VRAM
os.makedirs('outputs', exist_ok=True)

!python -m scripts.texture_i2tex --variant sd21 --image "{IMG}" --mesh "{MESH}" --save_dir outputs --save_name resultado --remove_bg

OUT = None
for root, dirs, fs in os.walk('outputs'):
    for f in fs:
        if f.endswith('.glb'):
            OUT = os.path.join(root, f)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024/1024, 2)) + ' MB')
      if OUT else '❌ no se generó — copiame TODO el error rojo de arriba')

## Celda 5 — Descargar
Probalo en https://gltf-viewer.donmccurdy.com — tu malla de siempre, ahora con el color de tu imagen alrededor de todo el modelo.

In [ ]:
from google.colab import files
import os
for root, dirs, fs in os.walk('/content/MV-Adapter/outputs'):
    for f in fs:
        if f.endswith('.glb'):
            files.download(os.path.join(root, f))

---
### Errores conocidos (ya contemplados, por si igual aparecen)
- **`FileNotFoundError: /content/MV-Adapter`** → la sesión se reinició sola → Celda 1 de nuevo.
- **`CUDA out of memory`** → raro con `--variant sd21`; cerrá otros notebooks abiertos y repetí la Celda 4. Si persiste: el mismo motor corre gratis online en https://huggingface.co/spaces/VAST-AI/MV-Adapter-Img2Texture (subís malla + imagen ahí, cero instalación).
- **`No module named 'X'`** → copiame el nombre X y te paso el `pip install` exacto.
- **Error `numpy`/`scipy` al importar** → corré `!pip install -q --force-reinstall "numpy>=2.3.0,<2.5" "scipy>=1.16.3"` y probá de nuevo (sin reiniciar).
- **Descarga de modelos lenta en la Celda 4** → normal la primera vez (varios GB desde Hugging Face sin token).
- Cualquier otra cosa roja: copiámela COMPLETA. 🧠